In [6]:
import os
import cv2
import numpy as np
from tensorflow.keras.models import load_model

In [7]:
input_folder = "extracted_words"

def clean_image(image, target_size=(40, 40)):
    """
    Cleans and preprocesses an image to reduce noise and adjust size.
    First, the image is resized to the target size, then it is converted to grayscale,
    blurred, and finally, binary thresholding is applied using the Otsu's method.
    """
    img_resized = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)  # Resize image
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)  # Apply Gaussian Blur
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)  # Apply Otsu's Thresholding
    return binary

def get_bounding_rect(contour):
    x, y, w, h = cv2.boundingRect(contour)
    return (x, y, w, h)

def sort_contours(contours):
    bounding_boxes = [get_bounding_rect(c) for c in contours]
    tolerance = 30
    lines = {}
    for i, (x, y, w, h) in enumerate(bounding_boxes):
        line_key = y // tolerance
        if line_key not in lines:
            lines[line_key] = []
        lines[line_key].append((x, y, w, h, i))
    sorted_lines = sorted(lines.items(), key=lambda item: item[0])
    contours_sorted = []
    for line_key, objects in sorted_lines:
        objects_sorted = sorted(objects, key=lambda obj: obj[0])
        for obj in objects_sorted:
            contours_sorted.append(contours[obj[4]])
    return contours_sorted, sorted_lines

In [8]:
for file_name in os.listdir(input_folder):
    if file_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
        image_path = os.path.join(input_folder, file_name)
        image = cv2.imread(image_path)
        if image is None:
            print(f"Erreur : Impossible de charger l'image {file_name}.")
            continue
        original_image = image.copy()
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 128, 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((1, 1), np.uint8)
        dilated = cv2.dilate(binary, kernel, iterations=2)
        contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours_sorted, sorted_lines = sort_contours(contours)
        base_name = os.path.splitext(file_name)[0]
        image_output_folder = os.path.join(input_folder, base_name)
        os.makedirs(image_output_folder, exist_ok=True)
        object_counter = 1
        for line_key, objects in sorted_lines:
            objects_sorted = sorted(objects, key=lambda obj: obj[0])
            for obj in objects_sorted:
                x, y, w, h, _ = obj
                extracted_object = original_image[:, x:x+w]
                # Calculate padding
                padding = int(0.15 * max(w, h))
                padded_object = cv2.copyMakeBorder(extracted_object, padding, padding, padding, padding, cv2.BORDER_CONSTANT, value=[255, 255, 255])
                object_path = os.path.join(image_output_folder, f"object_{object_counter}.jpg")
                cv2.imwrite(object_path, padded_object)
                object_counter += 1
                cv2.rectangle(original_image, (x, y), (x+w, y+h), (0, 255, 0), 2)
        annotated_image_path = os.path.join(image_output_folder, "image_with_boxes.jpg")
        #cv2.imwrite(annotated_image_path, original_image)